# Rodent AI - FOMO Training (MobileNetV2 alpha=0.5) on Google Colab

Trains a **FOMO** model inspired by Edge Impulse:
- Input: **96x96 grayscale**
- Backbone: **MobileNetV2 alpha=0.5** (pretrained ImageNet, fine-tuned)
- Output: **12x12 grid** (stride 8), each cell predicts rat / background
- Export: **TFLite int8**

## Before running
1. `fomo_dataset.zip` must be uploaded to your **Google Drive**.
2. Runtime -> Change runtime type -> **T4 GPU**.
3. Run all cells in order.
4. After training, use the printed `DEPLOY_THRESHOLD` in the ESP32 firmware.

## 1. Install dependencies

In [ ]:
!pip install -q tensorflow Pillow numpy scipy

import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))

## 2. Mount Google Drive and unzip dataset

In [ ]:
from google.colab import drive
# force_remount=True defeats Colab's stale Drive cache after you overwrite a file.
drive.mount('/content/drive', force_remount=True)

import os, zipfile, shutil

# ── EDIT this path if your zip is in a sub-folder ──────────────────────────
ZIP_PATH = '/content/drive/MyDrive/fomo_dataset.zip'
DEST     = '/content/fomo_dataset'
# ───────────────────────────────────────────────────────────────────────────

zip_mb = os.path.getsize(ZIP_PATH) / 1e6
print(f'Zip seen by Colab: {zip_mb:.1f} MB')

# Clean slate.
if os.path.exists(DEST):
    print('Removing old extracted data...')
    shutil.rmtree(DEST)
os.makedirs(DEST, exist_ok=True)

# Manual extraction that NORMALISES path separators. Windows PowerShell's
# Compress-Archive writes zip entries with backslashes ('training\\cur_0001.jpg'),
# which Linux/Colab treats as ONE flat filename instead of training/cur_0001.jpg —
# so the training/ and testing/ folders never appeared. Rewriting '\\' -> '/' and
# recreating the folders ourselves fixes it without re-zipping or re-uploading.
print('Unzipping (normalising Windows backslashes)...')
n_files = 0
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for info in z.infolist():
        norm = info.filename.replace('\\', '/')
        if norm.endswith('/'):
            os.makedirs(os.path.join(DEST, *norm.split('/')), exist_ok=True)
            continue
        target = os.path.join(DEST, *norm.split('/'))
        os.makedirs(os.path.dirname(target), exist_ok=True)
        with z.open(info) as src, open(target, 'wb') as dst:
            shutil.copyfileobj(src, dst)
        n_files += 1
print(f'Done. Extracted {n_files} files.\n')

# Robustness: if the splits are nested one level deeper, find them.
if not os.path.isdir(f'{DEST}/training'):
    for root, dirs, _ in os.walk(DEST):
        if 'training' in dirs and 'testing' in dirs:
            DEST = root
            print(f'Dataset root auto-detected at: {DEST}')
            break

# Quick sanity check — expect training: 4592, testing: 1140 for the new dataset.
for split in ('training', 'testing'):
    d = f'{DEST}/{split}'
    if os.path.isdir(d):
        imgs = [f for f in os.listdir(d) if f.endswith('.jpg')]
        print(f'{split}: {len(imgs)} images')
    else:
        print(f'{split}: MISSING (looked in {d})')


## 3. Parse labels → FOMO 12×12 grid targets

Each image maps to a **12×12×1** float label. Each rat marks the cell its centroid falls in **plus a 1-cell border** (a 3×3 block, `LABEL_SPREAD`) — this *label spreading* gives the network up to ~9× more positive signal than a single cell, lifting recall and calibrating the scores. Background images are all zeros.


In [ ]:
import json
import numpy as np
from pathlib import Path
from PIL import Image

IMG_SIZE     = 96    # FOMO input (96x96 px)
GRID_SIZE    = 12    # 96 / stride(8) = 12
STRIDE       = IMG_SIZE // GRID_SIZE   # 8
BILINEAR     = Image.Resampling.BILINEAR if hasattr(Image, 'Resampling') else Image.BILINEAR

# Target-building options
# USE_GAUSSIAN paints each rat as a soft Gaussian peak instead of a hard 3x3 block.
USE_GAUSSIAN = True
GAUSS_SIGMA  = 0.8
LABEL_SPREAD = 1

# Keep FALSE unless the firmware also letterboxes. The ESP32 firmware squashes
# QVGA -> 96x96, so training should do the same.
LETTERBOX = False


def prepare_image(im):
    """Grayscale PIL image -> (arr[96,96,1] float32, scale|None, pad_x, pad_y)."""
    ow, oh = im.size
    if LETTERBOX:
        s = IMG_SIZE / max(ow, oh)
        nw, nh = max(1, int(round(ow * s))), max(1, int(round(oh * s)))
        canvas = Image.new('L', (IMG_SIZE, IMG_SIZE), 0)
        px, py = (IMG_SIZE - nw) // 2, (IMG_SIZE - nh) // 2
        canvas.paste(im.resize((nw, nh), BILINEAR), (px, py))
        arr = np.array(canvas, dtype=np.float32) / 255.0
        return arr[..., np.newaxis], s, float(px), float(py)
    arr = np.array(im.resize((IMG_SIZE, IMG_SIZE), BILINEAR), dtype=np.float32) / 255.0
    return arr[..., np.newaxis], None, 0.0, 0.0


def _paint_gaussian(label, row, col, sigma=None):
    """Max-combine a Gaussian peak centred at grid cell (row, col) into label."""
    if sigma is None:
        sigma = GAUSS_SIGMA
    rad = int(np.ceil(3 * sigma))
    for r in range(max(0, row - rad), min(GRID_SIZE - 1, row + rad) + 1):
        for c in range(max(0, col - rad), min(GRID_SIZE - 1, col + rad) + 1):
            g = np.exp(-((r - row) ** 2 + (c - col) ** 2) / (2.0 * sigma * sigma))
            if g > label[r, c, 0]:
                label[r, c, 0] = g


def boxes_to_fomo_label(boxes_info, orig_w, orig_h, scale=None, pad_x=0.0, pad_y=0.0):
    """Edge Impulse box list -> (12,12,1) FOMO target grid."""
    label = np.zeros((GRID_SIZE, GRID_SIZE, 1), dtype=np.float32)
    for b in boxes_info:
        cx = b['x'] + b['width']  / 2.0
        cy = b['y'] + b['height'] / 2.0
        if scale is not None:
            cx96, cy96 = cx * scale + pad_x, cy * scale + pad_y
        else:
            cx96, cy96 = cx * IMG_SIZE / orig_w, cy * IMG_SIZE / orig_h
        col = int(min(max(cx96 // STRIDE, 0), GRID_SIZE - 1))
        row = int(min(max(cy96 // STRIDE, 0), GRID_SIZE - 1))
        if USE_GAUSSIAN:
            _paint_gaussian(label, row, col)
        else:
            r0, r1 = max(0, row - LABEL_SPREAD), min(GRID_SIZE - 1, row + LABEL_SPREAD)
            c0, c1 = max(0, col - LABEL_SPREAD), min(GRID_SIZE - 1, col + LABEL_SPREAD)
            label[r0:r1 + 1, c0:c1 + 1, 0] = 1.0
    return label


def load_split(split_dir, return_names=False):
    split_dir = Path(split_dir)
    data = json.loads((split_dir / 'bounding_boxes.labels').read_text())
    bb = data['boundingBoxes']
    images, labels, names = [], [], []
    skipped = 0
    for fname, boxes_info in bb.items():
        img_path = split_dir / fname
        if not img_path.exists():
            skipped += 1
            continue
        try:
            im = Image.open(img_path).convert('L')
            orig_w, orig_h = im.size
            arr, s, px, py = prepare_image(im)
        except Exception:
            skipped += 1
            continue
        label = boxes_to_fomo_label(boxes_info, orig_w, orig_h, scale=s, pad_x=px, pad_y=py)
        images.append(arr)
        labels.append(label)
        names.append(fname)
    if skipped:
        print(f'  [skipped {skipped} missing/corrupt files]')
    X = np.array(images, dtype=np.float32)
    Y = np.array(labels, dtype=np.float32)
    return (X, Y, names) if return_names else (X, Y)

print('Loading training set...')
X_train, Y_train, train_files = load_split(f'{DEST}/training', return_names=True)
print(f'  X={X_train.shape}  Y={Y_train.shape}')

print('Loading test set...')
X_test, Y_test, test_files = load_split(f'{DEST}/testing', return_names=True)
print(f'  X={X_test.shape}  Y={Y_test.shape}')

_tr_pos = (Y_train.max(axis=(1, 2, 3)) >= 0.5)
_te_pos = (Y_test.max(axis=(1, 2, 3)) >= 0.5)
print(f'\nTrain: {int(_tr_pos.sum())} positives (rat), {int((~_tr_pos).sum())} backgrounds')
print(f'Test:  {int(_te_pos.sum())} positives, {int((~_te_pos).sum())} backgrounds')
_mode = f'Gaussian(sigma={GAUSS_SIGMA})' if USE_GAUSSIAN else f'hard block(spread={LABEL_SPREAD})'
print(f'Targets: {_mode}, letterbox={LETTERBOX}')
print(f'Positive-cell mass (Y_train.mean): {Y_train.mean():.4f}  '
      f'(prior bias auto-adapts from this)')

## 4. Build FOMO model (MobileNetV2 alpha=0.5)

In [ ]:
import math
import tensorflow as tf
from tensorflow.keras import layers, Model

# alpha = MobileNetV2 width. 0.35 was very small; 0.5 gives the model more
# capacity to tell a rat from floor/wall texture (the main false-alarm problem),
# still tiny and ESP32-friendly. ImageNet weights exist for 0.35/0.5/0.75/1.0.
# Cost: model grows ~57KB -> ~90KB and latency ~0.8s -> ~1.1s. Revert to 0.35
# if the camera is too slow or flash is tight.
def build_fomo(input_shape=(96, 96, 1), alpha=0.5, pos_frac=0.05):
    inp = layers.Input(shape=input_shape, name='input_grayscale')

    # MobileNetV2 expects 3 channels — repeat the grayscale channel 3×.
    # Use a native Concatenate (NOT Lambda): Keras 3 can't infer a Lambda's
    # output shape here (raises NotImplementedError), and Lambda+tf.repeat is
    # also fragile for the int8 TFLite export later. Concatenate is both.
    x = layers.Concatenate(axis=-1, name='gray_to_rgb')([inp, inp, inp])

    # MobileNetV2's ImageNet weights were trained on inputs scaled to [-1, 1],
    # but our images arrive as [0, 1]. Without this remap the pretrained
    # features get the wrong range and barely activate — a second cause of the
    # collapse. Rescaling is a plain affine op, fully int8-TFLite safe.
    x = layers.Rescaling(scale=2.0, offset=-1.0, name='to_pm1')(x)  # [0,1] -> [-1,1]

    # MobileNetV2 backbone (ImageNet weights); width set by `alpha` above
    base = tf.keras.applications.MobileNetV2(
        input_shape=(input_shape[0], input_shape[1], 3),
        alpha=alpha,
        include_top=False,
        weights='imagenet'
    )

    # Find stride-8 feature layer (block_6 → 12×12 for 96×96 input)
    target_layer = None
    for lyr in base.layers:
        if 'block_6_expand_relu' in lyr.name:
            target_layer = lyr.name
            break
    # Fallback: first layer whose spatial output is 12×12
    if target_layer is None:
        for lyr in base.layers:
            out = lyr.output
            if hasattr(out, 'shape') and len(out.shape) == 4 and out.shape[1] == 12:
                target_layer = lyr.name
                break
    print(f'Backbone truncated at: {target_layer}')

    backbone = Model(
        inputs=base.input,
        outputs=base.get_layer(target_layer).output,
        name=f'mobilenetv2_a{int(round(alpha * 100)):03d}'
    )
    print(f'Feature map shape: {backbone.output_shape}')

    x = backbone(x)                                     # (B, 12, 12, C)

    # FOMO head
    x = layers.Conv2D(32, 1, padding='same',
                      activation='relu',    name='head_conv')(x)
    # Prior-bias init (RetinaNet trick): start the sigmoid near the true rat-cell
    # frequency (pos_frac) instead of 50%, so the background gradient doesn't
    # swamp training and collapse the model to all-empty. pos_frac is measured
    # from the (spread) labels — see the build call below — so it auto-adapts to
    # whatever LABEL_SPREAD is set to.
    pos_frac = min(max(pos_frac, 1e-4), 0.5)
    output_bias = tf.keras.initializers.Constant(-math.log((1.0 - pos_frac) / pos_frac))
    x = layers.Conv2D(1,  1, padding='same',
                      activation='sigmoid', bias_initializer=output_bias,
                      name='fomo_output')(x)  # (B, 12, 12, 1)

    return Model(inputs=inp, outputs=x, name='FOMO_rat')

# Prior bias from the actual positive-cell fraction of the (spread) labels.
POS_FRAC = float(Y_train.mean())
print(f'Output prior set from positive-cell fraction: {POS_FRAC:.4f}')
model = build_fomo(pos_frac=POS_FRAC)
model.summary(line_length=80)


## 5. Loss function and compile

Binary **focal loss** (γ=2, α=0.5) handles the imbalance (~13 positive cells out of 144 per image after label spreading). α=0.5 (up from the RetinaNet default 0.25) up-weights the rat cells. The base LR is **3e-4** — lower than the usual 1e-3 because spreading made the loss ~5× larger, and 1e-3 caused an early `val_loss` spike.


In [ ]:
def binary_focal_loss(gamma=2.0, alpha=0.5):
    """Focal loss — suppresses easy negatives, focuses on hard positives.

    alpha weights POSITIVE (rat) cells; (1-alpha) weights background. Kept at 0.5
    so the background term doesn't dominate the gradient (that had collapsed the
    model to all-empty). Raise toward 0.75 if recall is still too low.

    This smooth form ALSO accepts soft targets (the Gaussian heatmaps): with
    pt = 1 - |y_true - y_pred| and a linear alpha blend it is *identical* to the
    classic hard-label focal loss when y_true is 0/1 — so flipping USE_GAUSSIAN
    on or off needs no other change and can't regress the old behaviour.
    """
    def loss(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        pt = 1.0 - tf.abs(y_true - y_pred)                    # = y_pred if y_true=1, 1-y_pred if y_true=0
        at = alpha * y_true + (1.0 - alpha) * (1.0 - y_true)  # = alpha if y_true=1, 1-alpha if y_true=0
        return -tf.reduce_mean(at * tf.pow(1.0 - pt, gamma) * tf.math.log(pt))
    return loss

# LR kept at 3e-4 (Adam). Gaussian targets carry a bit LESS positive mass than
# the hard 3×3 blocks, so the focal-loss magnitude and its gradients are gentler,
# not larger — 3e-4 stays safe (the earlier val_loss spike came from the opposite:
# a mass INCREASE at 1e-3). ReduceLROnPlateau trims it further if needed.
model.compile(
    optimizer=tf.keras.optimizers.Adam(3e-4),
    loss=binary_focal_loss(gamma=2.0, alpha=0.5),
    metrics=['binary_accuracy']
)
print('Model compiled.')


## 6. Train

In [ ]:
EPOCHS    = 80          # a few more than before: augmentation makes each epoch
BATCH     = 32          # noisier, so give it longer (EarlyStopping still restores
SAVE_PATH = '/content/fomo_best.keras'   # the best epoch, so extra epochs are safe)

# Photometric train-time augmentation. Applies blur / brightness / gamma /
# contrast / noise to the INPUT image only (labels untouched -> geometry stays
# correct). It teaches the model to cope with the blurry, dark, low-quality real
# ESP32-CAM feed instead of relying on the clean, sharp web photos. Applied
# EQUALLY to rats and backgrounds, so it can't become a "blurry = background"
# shortcut. Set USE_TRAIN_AUG = False to train the old way if anything misbehaves.
USE_TRAIN_AUG = True

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        SAVE_PATH, save_best_only=True, monitor='val_loss', verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=20, restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=7, min_lr=1e-6, verbose=1
    ),
]

if USE_TRAIN_AUG:
    AUTOTUNE = tf.data.AUTOTUNE

    def _gaussian_kernel(size=5, sigma=1.0):
        ax = tf.range(-(size // 2), size // 2 + 1, dtype=tf.float32)
        xx, yy = tf.meshgrid(ax, ax)
        k = tf.exp(-(xx ** 2 + yy ** 2) / (2.0 * sigma ** 2))
        k = k / tf.reduce_sum(k)
        return tf.reshape(k, (size, size, 1, 1))

    _BLUR_K = _gaussian_kernel(5, 1.0)

    def _augment(x, y):
        # x: (96,96,1) in [0,1]; y unchanged -> photometric only, geometry safe.
        x = x * tf.random.uniform([], 0.7, 1.2)                        # brightness
        x = tf.pow(tf.clip_by_value(x, 0.0, 1.0),
                   tf.random.uniform([], 0.75, 1.5))                   # gamma (night/IR)
        m = tf.reduce_mean(x)
        x = (x - m) * tf.random.uniform([], 0.8, 1.2) + m             # contrast
        x = x + tf.random.normal(tf.shape(x), 0.0,
                                 tf.random.uniform([], 0.0, 0.05))     # sensor noise
        x = tf.cond(tf.random.uniform([]) < 0.4,                       # out-of-focus blur
                    lambda: tf.nn.depthwise_conv2d(x[tf.newaxis], _BLUR_K,
                                                   [1, 1, 1, 1], 'SAME')[0],
                    lambda: x)
        return tf.clip_by_value(x, 0.0, 1.0), y

    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, Y_train))
                .shuffle(2048, seed=42)
                .map(_augment, num_parallel_calls=AUTOTUNE)
                .batch(BATCH).prefetch(AUTOTUNE))
    val_ds = (tf.data.Dataset.from_tensor_slices((X_test, Y_test))
              .batch(BATCH).prefetch(AUTOTUNE))

    history = model.fit(train_ds, validation_data=val_ds,
                        epochs=EPOCHS, callbacks=callbacks, verbose=1)
else:
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_test, Y_test),
        epochs=EPOCHS,
        batch_size=BATCH,
        callbacks=callbacks,
        verbose=1
    )


## 7. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history.history['loss'],     label='train')
ax1.plot(history.history['val_loss'], label='val')
ax1.set_title('Focal Loss')
ax1.set_xlabel('Epoch')
ax1.legend()

ax2.plot(history.history['binary_accuracy'],     label='train')
ax2.plot(history.history['val_binary_accuracy'], label='val')
ax2.set_title('Binary Accuracy')
ax2.set_xlabel('Epoch')
ax2.legend()

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=120)
plt.show()

## 8. Evaluate - Precision / Recall / F1 on test set

In [ ]:
CONF_THRESHOLD = 0.4   # balanced default from the previous object-level sweep
GT_POS_LEVEL   = 0.5   # a ground-truth cell counts as rat if its target >= this
THRESHOLDS     = np.round(np.arange(0.05, 0.86, 0.05), 2)

preds    = model.predict(X_test, batch_size=32, verbose=0)   # (N, 12, 12, 1)
gt_bin   = (Y_test >= GT_POS_LEVEL).astype(np.float32)
pred_bin = (preds  >= CONF_THRESHOLD).astype(np.float32)


def cell_prf(pred_bin, gt_bin):
    tp = int(np.sum((pred_bin == 1) & (gt_bin == 1)))
    fp = int(np.sum((pred_bin == 1) & (gt_bin == 0)))
    fn = int(np.sum((pred_bin == 0) & (gt_bin == 1)))
    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return tp, fp, fn, precision, recall, f1


tp, fp, fn, precision, recall, f1 = cell_prf(pred_bin, gt_bin)
print(f'Threshold : {CONF_THRESHOLD}')
print(f'TP={tp}  FP={fp}  FN={fn}')
print(f'Precision : {precision:.3f}')
print(f'Recall    : {recall:.3f}')
print(f'F1        : {f1:.3f}')

bg_mask = gt_bin.sum(axis=(1, 2, 3)) == 0
fp_images = int((pred_bin[bg_mask].sum(axis=(1, 2, 3)) > 0).sum())
print(f'\nFalse-positive images: {fp_images} / {int(bg_mask.sum())} backgrounds')

CELL_RESULTS = []
print('\nCell-level sweep')
print('thr     P      R      F1')
for thr in THRESHOLDS:
    pb = (preds >= thr).astype(np.float32)
    _tp, _fp, _fn, _p, _r, _f = cell_prf(pb, gt_bin)
    CELL_RESULTS.append({'thr': float(thr), 'tp': _tp, 'fp': _fp, 'fn': _fn,
                         'precision': _p, 'recall': _r, 'f1': _f})
    print(f'{thr:0.2f}  {_p:.3f}  {_r:.3f}  {_f:.3f}')

best_cell = max(CELL_RESULTS, key=lambda r: r['f1'])
print(f"\nBest cell-level F1 = {best_cell['f1']:.3f} at threshold {best_cell['thr']:.2f}")

In [ ]:
# Object-level (per-rat) evaluation
# This is the deployment metric: threshold heatmap -> connected blobs -> blob centroid.
from scipy import ndimage
from pathlib import Path
from PIL import Image
import csv

MATCH_DIST = 1.5   # cells; a detection within this of a GT centroid = a hit

try:
    preds
except NameError:
    preds = model.predict(X_test, batch_size=32, verbose=0)


def blob_centroids(grid_2d, thr):
    mask = grid_2d >= thr
    lbl, n = ndimage.label(mask)  # default 2D structure matches firmware 4-connectivity
    if n == 0:
        return []
    return ndimage.center_of_mass(mask, lbl, range(1, n + 1))


def match_objects_for_image(pred_grid, gt_grid, thr, match_dist=MATCH_DIST):
    gt = blob_centroids(gt_grid, 0.5)
    pr = blob_centroids(pred_grid, thr)
    used = [False] * len(gt)
    tp = fp = 0
    for (pr_r, pr_c) in pr:
        best, best_d = -1, 1e9
        for j, (g_r, g_c) in enumerate(gt):
            if used[j]:
                continue
            dd = ((pr_r - g_r) ** 2 + (pr_c - g_c) ** 2) ** 0.5
            if dd < best_d:
                best, best_d = j, dd
        if best >= 0 and best_d <= match_dist:
            used[best] = True
            tp += 1
        else:
            fp += 1
    fn = used.count(False)
    return tp, fp, fn, len(gt), len(pr)


def object_prf(preds, Y, thr, match_dist=MATCH_DIST):
    TP = FP = FN = 0
    for i in range(len(preds)):
        tp, fp, fn, _, _ = match_objects_for_image(
            preds[i, :, :, 0], Y[i, :, :, 0], thr, match_dist)
        TP += tp
        FP += fp
        FN += fn
    P = TP / (TP + FP + 1e-9)
    R = TP / (TP + FN + 1e-9)
    F = 2 * P * R / (P + R + 1e-9)
    return TP, FP, FN, P, R, F


print('Per-object detection (predicted blob centroid -> GT rat centroid, tol=1.5 cells)')
print('thr     TP    FP    FN      P      R      F1')
OBJECT_RESULTS = []
for thr in THRESHOLDS:
    TP, FP, FN, P, R, F = object_prf(preds, Y_test, float(thr))
    OBJECT_RESULTS.append({'thr': float(thr), 'tp': TP, 'fp': FP, 'fn': FN,
                           'precision': P, 'recall': R, 'f1': F})
    print(f'{thr:0.2f}  {TP:5} {FP:5} {FN:5}   {P:.3f}  {R:.3f}  {F:.3f}')

best_obj = max(OBJECT_RESULTS, key=lambda r: (r['f1'], r['recall']))
DEPLOY_THRESHOLD = float(best_obj['thr'])
print(f"\nBest object-level F1 = {best_obj['f1']:.3f} at threshold {DEPLOY_THRESHOLD:.2f}")
print(f'Use this in firmware: const float DETECT_THRESHOLD = {DEPLOY_THRESHOLD:.2f}f;')

Path('/content/best_threshold.txt').write_text(
    f"DETECT_THRESHOLD={DEPLOY_THRESHOLD:.2f}\n"
    f"precision={best_obj['precision']:.4f}\n"
    f"recall={best_obj['recall']:.4f}\n"
    f"f1={best_obj['f1']:.4f}\n"
)

# Export an error-mining report. Add the high-FP rows as background/hard negatives,
# and add the high-FN rows as missed-rat positives after checking labels.
error_rows = []
for i in range(len(preds)):
    tp, fp, fn, gt_count, pred_count = match_objects_for_image(
        preds[i, :, :, 0], Y_test[i, :, :, 0], DEPLOY_THRESHOLD)
    if fp or fn:
        name = test_files[i] if 'test_files' in globals() and i < len(test_files) else f'test_{i:05d}'
        error_rows.append({
            'index': i, 'file': name, 'gt_objects': gt_count, 'pred_objects': pred_count,
            'tp': tp, 'fp': fp, 'fn': fn,
        })

error_rows.sort(key=lambda r: (r['fn'] + r['fp'], r['fn'], r['fp']), reverse=True)
ERROR_REPORT_PATH = Path('/content/detection_error_review.csv')
with ERROR_REPORT_PATH.open('w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['index', 'file', 'gt_objects', 'pred_objects', 'tp', 'fp', 'fn'])
    writer.writeheader()
    writer.writerows(error_rows)
print(f'Wrote error-mining report: {ERROR_REPORT_PATH} ({len(error_rows)} rows)')

review_dir = Path('/content/error_review')
review_dir.mkdir(exist_ok=True)
for row in error_rows[:80]:
    i = row['index']
    safe_name = Path(row['file']).stem.replace(' ', '_')[:40]
    out = Image.fromarray((X_test[i, :, :, 0] * 255).clip(0, 255).astype(np.uint8))
    out.save(review_dir / f"{i:05d}_{safe_name}_fn{row['fn']}_fp{row['fp']}.png")
print(f'Saved up to 80 review thumbnails in: {review_dir}')

## 9. Export TFLite int8 + Check Quantized Performance

Quantises weights **and** activations to int8 using a larger representative set that includes both rat and background images. Then evaluates the exported int8 model with the same object-level metric before deployment.

In [ ]:
# Restore best weights
model.load_weights(SAVE_PATH)

# Representative dataset for int8 calibration. Keep backgrounds/hard negatives in
# the calibration set so int8 activation ranges do not overfit to rat-only images.
REP_SAMPLES = min(300, len(X_train))
rng = np.random.default_rng(42)
pos_idx = np.flatnonzero(Y_train.max(axis=(1, 2, 3)) >= 0.5)
bg_idx  = np.flatnonzero(Y_train.max(axis=(1, 2, 3)) < 0.5)

n_bg = min(len(bg_idx), max(1, REP_SAMPLES // 3))
n_pos = min(len(pos_idx), REP_SAMPLES - n_bg)
rep_indices = []
if n_pos:
    rep_indices.extend(rng.choice(pos_idx, size=n_pos, replace=False).tolist())
if n_bg:
    rep_indices.extend(rng.choice(bg_idx, size=n_bg, replace=False).tolist())
if len(rep_indices) < REP_SAMPLES:
    remaining = np.setdiff1d(np.arange(len(X_train)), np.array(rep_indices, dtype=int))
    fill = min(len(remaining), REP_SAMPLES - len(rep_indices))
    if fill:
        rep_indices.extend(rng.choice(remaining, size=fill, replace=False).tolist())
rng.shuffle(rep_indices)
print(f'Representative samples: {len(rep_indices)} ({n_pos} positive, {n_bg} background)')


def representative_dataset():
    for i in rep_indices:
        yield [X_train[i:i+1].astype(np.float32)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations          = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type   = tf.int8
converter.inference_output_type  = tf.int8

tflite_model = converter.convert()

TFLITE_PATH = '/content/fomo_rat_int8.tflite'
with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)
print(f'TFLite int8: {TFLITE_PATH}  ({len(tflite_model)//1024} KB)')


def _quantize(x, details):
    dtype = details['dtype']
    if dtype == np.float32:
        return x.astype(np.float32)
    scale, zp = details['quantization']
    if scale == 0:
        raise ValueError('Input quantization scale is 0')
    q = np.round(x / scale + zp)
    info = np.iinfo(dtype)
    return np.clip(q, info.min, info.max).astype(dtype)


def _dequantize(y, details):
    if details['dtype'] == np.float32:
        return y.astype(np.float32)
    scale, zp = details['quantization']
    return (y.astype(np.float32) - zp) * scale


def predict_tflite(model_path, X, max_items=None):
    intr = tf.lite.Interpreter(model_path=model_path)
    in_details = intr.get_input_details()[0]
    out_details = intr.get_output_details()[0]
    expected = list(in_details['shape'])
    if expected[0] != 1:
        intr.resize_tensor_input(in_details['index'], [1, IMG_SIZE, IMG_SIZE, 1])
    intr.allocate_tensors()
    in_details = intr.get_input_details()[0]
    out_details = intr.get_output_details()[0]

    n = len(X) if max_items is None else min(len(X), max_items)
    outs = []
    for i in range(n):
        sample = X[i:i+1].astype(np.float32)
        intr.set_tensor(in_details['index'], _quantize(sample, in_details))
        intr.invoke()
        outs.append(_dequantize(intr.get_tensor(out_details['index']), out_details))
    return np.concatenate(outs, axis=0)

print('\nEvaluating int8 TFLite model on the test set...')
tflite_preds = predict_tflite(TFLITE_PATH, X_test)
TFLITE_RESULTS = []
print('thr     TP    FP    FN      P      R      F1')
for thr in THRESHOLDS:
    TP, FP, FN, P, R, F = object_prf(tflite_preds, Y_test, float(thr))
    TFLITE_RESULTS.append({'thr': float(thr), 'tp': TP, 'fp': FP, 'fn': FN,
                           'precision': P, 'recall': R, 'f1': F})
    print(f'{thr:0.2f}  {TP:5} {FP:5} {FN:5}   {P:.3f}  {R:.3f}  {F:.3f}')

best_int8 = max(TFLITE_RESULTS, key=lambda r: (r['f1'], r['recall']))
INT8_DEPLOY_THRESHOLD = float(best_int8['thr'])
print(f"\nBest int8 object-level F1 = {best_int8['f1']:.3f} at threshold {INT8_DEPLOY_THRESHOLD:.2f}")
print(f'Prefer this firmware value after export: const float DETECT_THRESHOLD = {INT8_DEPLOY_THRESHOLD:.2f}f;')

report = Path('/content/tflite_eval_report.txt')
report.write_text(
    f"INT8_DETECT_THRESHOLD={INT8_DEPLOY_THRESHOLD:.2f}\n"
    f"precision={best_int8['precision']:.4f}\n"
    f"recall={best_int8['recall']:.4f}\n"
    f"f1={best_int8['f1']:.4f}\n"
)

## 10. Save to Google Drive + download

In [ ]:
import shutil
from pathlib import Path
from google.colab import files

try:
    DEPLOY_THRESHOLD
except NameError:
    DEPLOY_THRESHOLD = 0.4

try:
    INT8_DEPLOY_THRESHOLD
except NameError:
    INT8_DEPLOY_THRESHOLD = DEPLOY_THRESHOLD

drive_out = Path('/content/drive/MyDrive/rodent_fomo_model')
drive_out.mkdir(exist_ok=True)

shutil.copy(SAVE_PATH,                      drive_out / 'fomo_best.keras')
shutil.copy(TFLITE_PATH,                    drive_out / 'fomo_rat_int8.tflite')
shutil.copy('/content/training_curves.png', drive_out / 'training_curves.png')

for extra in ('/content/best_threshold.txt',
              '/content/tflite_eval_report.txt',
              '/content/detection_error_review.csv'):
    p = Path(extra)
    if p.exists():
        shutil.copy(p, drive_out / p.name)

Path('/content/deployment_threshold.h').write_text(
    f'const float DETECT_THRESHOLD = {INT8_DEPLOY_THRESHOLD:.2f}f;\n'
)
shutil.copy('/content/deployment_threshold.h', drive_out / 'deployment_threshold.h')

print('Saved to Drive:', drive_out)
print(f'Firmware threshold from int8 eval: {INT8_DEPLOY_THRESHOLD:.2f}')

# Direct download to your computer
files.download(TFLITE_PATH)

## Notes on ESP32-CAM deployment

The exported `fomo_rat_int8.tflite` is a **custom model**.

### Direct firmware deployment
1. Copy the new `fomo_rat_int8.tflite` into `firmware/rodent_fomo_webserver/`.
2. Regenerate `fomo_model_data.h` from that `.tflite`.
3. Set `DETECT_THRESHOLD` in the firmware to the value printed by the int8 evaluation cell.
4. Keep camera preprocessing matched to training: QVGA frame squashed to 96x96 grayscale with bilinear resize.

### How the FOMO output maps to detections
- Output tensor: **12x12x1** int8 heatmap.
- Dequantise with output tensor scale and zero point.
- Threshold hot cells, group touching cells into blobs, and use each blob centroid as one rat detection.